# 02 — Score TMDB Movies with NRC VAD + Emotion Intensity

Loads the full TMDB dataset (~930K movies), pre-scores every unique keyword
using the pickled NRC lookup dicts from `01_lexicons.ipynb`, then aggregates
to movie level and exports to CSV for Kaggle upload.

**Prerequisites:** run `01_lexicons.ipynb` first to generate `data/lexicons/nrc_lookups.pkl`.

In [1]:
import os
import pickle
import re
import time
import zipfile
from pathlib import Path

import kagglehub
import pandas as pd
from dotenv import load_dotenv
from nltk.stem import WordNetLemmatizer
import nltk

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

load_dotenv()

DATASET  = 'asaniczka/tmdb-movies-dataset-2023-930k-movies'
NROWS    = None   # None = full ~930K dataset; set to 10_000 for quick tests

_lemmatizer = WordNetLemmatizer()

def _tokenize(keyword: str) -> list:
    return [t for t in re.split(r'[\s\-_/]+', keyword.lower()) if t.isalpha()]

def _lemma_candidates(tok: str) -> list:
    seen, out = set(), []
    for pos in ('n', 'v', 'a'):
        lemma = _lemmatizer.lemmatize(tok, pos=pos)
        if lemma not in seen:
            seen.add(lemma)
            out.append(lemma)
    if tok not in seen:
        out.append(tok)
    return out

In [2]:
# ── Load TMDB dataset ─────────────────────────────────────────────────────
t0 = time.time()
base = Path(kagglehub.dataset_download(DATASET))
csvs = list(base.rglob('*.csv')) or []
if not csvs:
    zips = sorted(base.rglob('*.zip'))
    import zipfile
    with zipfile.ZipFile(zips[0]) as z:
        csvs = [z.extract(n, base) for n in z.namelist() if n.endswith('.csv')]

pandas_kwargs = {'nrows': NROWS} if NROWS else {}
df = pd.read_csv(sorted(csvs)[0], **pandas_kwargs)
print(f'Loaded {len(df):,} rows in {time.time()-t0:.1f}s')

Loaded 1,382,594 rows in 5.6s


In [3]:
# ── Load pickled NRC lookup dicts ─────────────────────────────────────────
with open('data/lexicons/nrc_lookups.pkl', 'rb') as f:
    nrc = pickle.load(f)

intensity_lookup = nrc['intensity_lookup']
vad_lookup       = nrc['vad_lookup']
EMOTIONS         = nrc['EMOTIONS']
VAD_DIMS         = nrc['VAD_DIMS']

print(f'intensity_lookup: {len(intensity_lookup):,} words')
print(f'vad_lookup:       {len(vad_lookup):,} words')

intensity_lookup: 5,891 words
vad_lookup:       19,971 words


In [4]:
# ── Pre-score all unique keywords ─────────────────────────────────────────
# Full-phrase lookup first (hits MWEs like "child abuse" directly),
# then falls back to per-token unigram lookup with lemmatization.
t0 = time.time()

unique_keywords = sorted(
    {kw.strip() for s in df['keywords'].dropna().astype(str) for kw in s.split(',') if kw.strip()}
)
print(f'Unique keywords: {len(unique_keywords):,}')


def _score_keyword(keyword, intensity_lookup, vad_lookup, emotions, vad_dims):
    # 1. Try full phrase directly (catches MWEs like "child abuse", "space marine")
    if keyword in vad_lookup:
        v_rows = [vad_lookup[keyword]]
    else:
        # 2. Tokenize and look up each token with lemmatization
        tokens = _tokenize(keyword)
        v_rows = []
        for tok in tokens:
            for lemma in _lemma_candidates(tok):
                if lemma in vad_lookup:
                    v_rows.append(vad_lookup[lemma]); break

    # Emotion intensity — always token-level
    tokens = _tokenize(keyword)
    e_rows = []
    for tok in tokens:
        for lemma in _lemma_candidates(tok):
            if lemma in intensity_lookup:
                e_rows.append(intensity_lookup[lemma]); break

    emotion_scores = ({e: sum(r[e] for r in e_rows)/len(e_rows) for e in emotions}
                      if e_rows else {e: 0.0 for e in emotions})
    vad_scores = ({d: sum(r[d] for r in v_rows)/len(v_rows) for d in vad_dims}
                  if v_rows else {d: float('nan') for d in vad_dims})
    return {**emotion_scores, **vad_scores}


kw_scores = {
    kw: _score_keyword(kw, intensity_lookup, vad_lookup, EMOTIONS, VAD_DIMS)
    for kw in unique_keywords
}

elapsed = time.time() - t0
phrase_hits = sum(1 for kw in unique_keywords if kw in vad_lookup)
print(f'Scored {len(unique_keywords):,} keywords in {elapsed:.2f}s')
print(f'Direct phrase hits: {phrase_hits:,} ({phrase_hits/len(unique_keywords):.1%} of unique keywords)')

Unique keywords: 67,908


Pre-scored 67,908 keywords in 2.7s


In [5]:
# ── Aggregate to movie level (optimised) ─────────────────────────────────
t0 = time.time()

def score_movie(keywords_str):
    if not isinstance(keywords_str, str) or not keywords_str.strip():
        return {**{e: 0.0 for e in EMOTIONS}, **{d: float('nan') for d in VAD_DIMS}}
    kws = [kw.strip() for kw in keywords_str.split(',') if kw.strip()]

    e_rows = [kw_scores[kw] for kw in kws if kw in kw_scores]
    emotion_means = {}
    for e in EMOTIONS:
        vals = [r[e] for r in e_rows if r[e] > 0]
        emotion_means[e] = sum(vals) / len(vals) if vals else 0.0

    vad_means = {}
    for d in VAD_DIMS:
        vals = [r[d] for r in e_rows if r[d] == r[d]]  # NaN-safe
        vad_means[d] = sum(vals) / len(vals) if vals else float('nan')

    return {**emotion_means, **vad_means}


# pd.DataFrame(tolist()) is ~10x faster than .apply(pd.Series) on large frames
scores_df = pd.DataFrame(df['keywords'].map(score_movie).tolist())
scores_df['sentiment'] = scores_df['valence'].apply(
    lambda v: 'positive' if v > 0 else ('negative' if v < 0 else 'neutral')
    if v == v else 'unknown'
)

elapsed = time.time() - t0
print(f'Scored {len(df):,} movies in {elapsed:.1f}s ({len(df)/elapsed:,.0f} movies/sec)')
print()
print('Sentiment distribution:')
print(scores_df['sentiment'].value_counts().to_string())

Scored 1,382,594 movies in 3.3s (413,682 movies/sec)

Sentiment distribution:
sentiment
unknown     1061699
positive     223080
negative      97153
neutral         662


In [6]:
# ── Classify keywords by valence ────────────────────────────────────────
# Buckets each movie's keywords by their individual NRC VAD valence score.
# positive: valence > 0.5  |  negative: valence < 0.5
# neutral:  valence == 0.5 |  unknown: no VAD coverage

def classify_keywords(keywords_str):
    if not isinstance(keywords_str, str) or not keywords_str.strip():
        return {'positive_keywords': '', 'negative_keywords': '', 'neutral_keywords': '', 'unknown_keywords': ''}
    kws = [kw.strip() for kw in keywords_str.split(',') if kw.strip()]
    pos, neg, unk = [], [], []
    for kw in kws:
        v = kw_scores.get(kw, {}).get('valence', float('nan'))
        if v != v:      unk.append(kw)   # NaN = no VAD coverage
        elif v >= 0:    pos.append(kw)
        else:           neg.append(kw)
    return {
        'positive_keywords': ', '.join(pos),
        'negative_keywords': ', '.join(neg),
        'unknown_keywords':  ', '.join(unk),
    }

kw_groups = pd.DataFrame(df['keywords'].map(classify_keywords).tolist())
print('Keyword group sample:')
sample = df[['title', 'keywords']].join(kw_groups)
display(sample[sample['positive_keywords'] != ''].head(3)[['title', 'positive_keywords', 'negative_keywords', 'unknown_keywords']])


Keyword group sample:


,title,positive_keywords,negative_keywords,unknown_keywords
0,Inception,"rescue, mission, dream, airplane, virtual real...","kidnapping, spy, manipulation, car crash, heist","paris, france, los angeles, california"
1,Interstellar,"rescue, future, spacecraft, race against time,...","time warp, wormhole, famine, robot, time-manip...","nasa, dystopia, 2060s"
2,The Dark Knight,"joker, secret identity, superhero, anti hero, ...","chaos, crime fighter, scarecrow, vigilante, or...",sadism


In [7]:
import tracemalloc
import gc

# ── Performance benchmark ─────────────────────────────────────────────────
print('=' * 55)
print('PERFORMANCE BENCHMARK')
print('=' * 55)

# Keyword pre-scoring
t_kw_start = time.perf_counter()
_ = {
    kw: _score_keyword(kw, intensity_lookup, vad_lookup, EMOTIONS, VAD_DIMS)
    for kw in unique_keywords
}
t_kw = time.perf_counter() - t_kw_start
print(f'Keyword scoring:   {len(unique_keywords):>8,} keywords  {t_kw:.2f}s  ({len(unique_keywords)/t_kw:,.0f} kw/sec)')

# Movie scoring (with memory tracking)
gc.collect()
tracemalloc.start()
t_mv_start = time.perf_counter()
scores_list = df['keywords'].map(score_movie).tolist()
t_mv = time.perf_counter() - t_mv_start
_, peak_bytes = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f'Movie scoring:     {len(df):>8,} movies    {t_mv:.2f}s  ({len(df)/t_mv:,.0f} movies/sec)')
print(f'Peak memory:       {peak_bytes/1e6:>8.1f} MB')

# Coverage stats
has_valence  = scores_df['valence'].notna().sum()
has_emotion  = (scores_df[EMOTIONS].sum(axis=1) > 0).sum()
no_keywords  = df['keywords'].isna().sum()
print()
print('COVERAGE')
print('-' * 55)
print(f'Movies total:      {len(df):>8,}')
print(f'No keywords:       {no_keywords:>8,}  ({100*no_keywords/len(df):.1f}%)')
print(f'VAD valence match: {has_valence:>8,}  ({100*has_valence/len(df):.1f}%)')
print(f'Emotion signal:    {has_emotion:>8,}  ({100*has_emotion/len(df):.1f}%)')
print(f'Unique keywords:   {len(unique_keywords):>8,}')
print('=' * 55)
del scores_list

PERFORMANCE BENCHMARK


Keyword scoring:     67,908 keywords  1.44s  (47,114 kw/sec)


Movie scoring:     1,382,594 movies    25.19s  (54,892 movies/sec)
Peak memory:          827.6 MB

COVERAGE
-------------------------------------------------------
Movies total:      1,382,594
No keywords:       1,037,355  (75.0%)
VAD valence match:  320,895  (23.2%)
Emotion signal:     230,300  (16.7%)
Unique keywords:     67,908


In [8]:
# ── Sanity check ─────────────────────────────────────────────────────────
display_cols = ['title', 'keywords', 'valence', 'arousal', 'dominance', 'sentiment']
sanity_df = df[['title', 'keywords']].join(scores_df[['valence', 'arousal', 'dominance', 'sentiment']])
print('Most positive valence:')
display(sanity_df.nlargest(5, 'valence')[display_cols])
print()
print('Most negative valence:')
display(sanity_df.nsmallest(5, 'valence')[display_cols])

Most positive valence:


,title,keywords,valence,arousal,dominance,sentiment
5811,Little Italy,love,1.0,0.519,0.673,positive
5827,A Perfect Plan,"france, love",1.0,0.519,0.673,positive
7641,Ma che bella sorpresa,love,1.0,0.519,0.673,positive
8069,Skin,"skinhead, love",1.0,0.519,0.673,positive
8627,A Short Film About Love,love,1.0,0.519,0.673,positive



Most negative valence:


,title,keywords,valence,arousal,dominance,sentiment
82673,The Johnsons,"nightmare, menstruation",0.005,0.81,0.436,negative
91267,Dream House Nightmare,nightmare,0.005,0.81,0.436,negative
100909,Shhh...,nightmare,0.005,0.81,0.436,negative
126530,Bed Time,nightmare,0.005,0.81,0.436,negative
154645,Ingenium,nightmare,0.005,0.81,0.436,negative


In [9]:
import json

OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / 'movie_vad_scores.csv'

# Column order: identity → emotion scores → remaining TMDB metadata
# This ensures the Kaggle preview shows the most meaningful columns first
IDENTITY   = ['id', 'title', 'keywords']
KW_GROUPS  = ['positive_keywords', 'negative_keywords', 'unknown_keywords']
SCORED     = ['sentiment', 'valence', 'arousal', 'dominance'] + EMOTIONS
TMDB_REST  = [c for c in df.columns if c not in IDENTITY + ['keywords']]

out_df = pd.concat([df[IDENTITY], kw_groups[KW_GROUPS], scores_df[SCORED], df[TMDB_REST]], axis=1)
out_df.to_csv(OUTPUT_CSV, index=False)

size_mb = OUTPUT_CSV.stat().st_size / 1e6
print(f'Wrote {len(out_df):,} rows x {len(out_df.columns)} cols -> {OUTPUT_CSV}  ({size_mb:.1f} MB)')
print(f'Column order: {list(out_df.columns[:8])} ... {list(out_df.columns[-4:])}')

# Write dataset metadata
print('Metadata ready at output/dataset-metadata.json')

Wrote 1,382,594 rows x 39 cols -> output/movie_vad_scores.csv  (641.0 MB)
Column order: ['id', 'title', 'keywords', 'positive_keywords', 'negative_keywords', 'unknown_keywords', 'sentiment', 'valence'] ... ['genres', 'production_companies', 'production_countries', 'spoken_languages']
Metadata ready at output/dataset-metadata.json
